In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("me/Linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

   
Contact
patrick@patrickvanvilsteren
.eu
www.linkedin.com/in/pvanvilsteren
(LinkedIn)
Top Skills
Windows Server
Servers
ITIL
Certifications
Managing and Maintaining a
Windows Server 2003 Environment
70-290
Microsoft Certified: Azure
Fundamentals
ITIL certified V3
MCDST
Patrick Van Vilsteren
Technology Services Engineer 2
Zwolle, Overijssel, Netherlands
Summary
I'm a hard working professional, and I have a flexible attitude. I have
done a few improvement project for helpdesks, but I really miss
the technical side of my profession. In February of this year I took
my CCNA exam and passed. Now I'm busy with my Windows 7
certification. And plan on doing more!
Special experiences:
4 weeks in Hungary for the transition of the T-Mobile service desk.
I had to report back to management and evaluate people on their
language skills. Furthermore I had to guide the employees there in
the processes.
A few days in the US to meet my new co-workers and to familiar
myself with the processes at Arizona

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Patrick van Vilsteren"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [9]:
system_prompt

"You are acting as Patrick van Vilsteren. You are answering questions on Patrick van Vilsteren's website, particularly questions related to Patrick van Vilsteren's career, background, skills and experience. Your responsibility is to represent Patrick van Vilsteren for interactions on the website as faithfully as possible. You are given a summary of Patrick van Vilsteren's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Patrick van Vilsteren. I am a Technical Engineer. I am learning AI and study data Engineering\nMy hobby's are Flying DCS World, playing tennis and learning new things.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\npatrick@patrickvanvilsteren\n.eu\nwww.linkedin.com/in/pvanvilsteren\n(LinkedIn)\nTop Skills\nWindows Server\nServers\nITIL\nCertifications\nManaging and Maintaini

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
    return response.choices[0].message.content

In [12]:
# Create a Pydentic model for the evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [13]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n Summary:\n{summary}\n\n## Linkedin Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with wheter the response is acceptable and your feedback."


In [14]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt 

In [ ]:
import os
